# 🚀 00 — Environment Setup
## Ekegusii-LLM-Translation · Kineses Cloud / Base Jupyter

> **Run this notebook ONCE after cloning in `start.ipynb`.**

This notebook:
1. Sets working directory to project root
2. Verifies all pre-installed conda packages
3. Confirms GPU (NVIDIA A100-SXM4-80GB)
4. Verifies Master Corpus & 0% data leakage

---
| Node Spec | Value |
|-----------|-------|
| Python | 3.11.6 (conda-forge) |
| CPU Cores | 22 |
| RAM | 117.9 GB |
| Disk | 967.64 GB |
| GPU | NVIDIA A100-SXM4-80GB (85.1 GB VRAM) |

In [ ]:
# ============================================================
# PATH BOOSTER — Works on Kineses / Jupyter / Colab / Kaggle
# Handles FileNotFoundError when kernel CWD no longer exists.
# ============================================================
import os, sys

# Step 1: Safely get current directory (may throw on Kineses)
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

# Step 2: Always try the known Kineses project path first
kineses_proj = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(kineses_proj):
    os.chdir(kineses_proj)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Step 3: Add project root to Python path
proj_root = os.getcwd()
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

print(f'✅ Working Directory : {os.getcwd()}')
print(f'✅ Python Kernel     : {sys.executable}')

In [ ]:
# ==============================================================
# CELL 1 — Set working directory & sys.path
# Run this first every time you open a new Jupyter session
# ==============================================================
import os, sys

home     = os.path.expanduser("~")
proj_dir = os.path.join(home, "Ekegusii-LLM-Translation-main")

if not os.path.isdir(proj_dir):
    raise RuntimeError(
        f"\n❌ Repository not found at: {proj_dir}\n"
        "   Please run start.ipynb (Cell 2 — FRESH CLONE) first!"
    )

os.chdir(proj_dir)
if proj_dir not in sys.path:
    sys.path.insert(0, proj_dir)

print(f"✅ Working Directory : {os.getcwd()}")
print(f"✅ Python Kernel     : {sys.executable}")

In [ ]:
# ==============================================================
# CELL 2 — Verify pre-installed conda packages (no pip needed)
# Kineses conda env already has all packages pre-installed.
# ==============================================================
print("=" * 60)
print("Verifying conda pre-installed packages...")
print("=" * 60)

import importlib

REQUIRED = [
    "torch", "pandas", "numpy", "matplotlib",
    "transformers", "datasets", "evaluate",
    "peft", "trl", "bitsandbytes",
    "sacrebleu", "accelerate", "tokenizers",
    "rich", "tqdm", "scipy", "sklearn",
]

all_ok = True
for mod in REQUIRED:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "installed")
        print(f"  ✅  {mod:<20} {ver}")
    except ImportError:
        print(f"  ❌  {mod:<20} NOT FOUND")
        all_ok = False

if all_ok:
    print("\n✅ All packages verified!")
else:
    print("\n⚠️  Some packages missing. Contact your Kineses admin.")

In [ ]:
# ==============================================================
# CELL 3 — Verify GPU hardware
# ==============================================================
import torch

print("=" * 60)
print("GPU Hardware Check")
print("=" * 60)
print(f"  PyTorch Version  : {torch.__version__}")
print(f"  CUDA Available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Name         : {torch.cuda.get_device_name(0)}")
    print(f"  GPU VRAM         : {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute Cap.     : {props.major}.{props.minor}")
    print(f"  CUDA Device      : cuda:{torch.cuda.current_device()}")
    print("\n  ✅ A100 Ready for QLoRA fine-tuning!")
else:
    print("\n  ⚠️  No GPU — training will run on CPU only.")

In [ ]:
# ==============================================================
# CELL 4 — Verify Master Corpus & 0% Data Leakage
# ==============================================================
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.integrity import DataLeakageChecker

print("=" * 60)
print("Master Corpus Verification")
print("=" * 60)

manager = MasterCorpusManager()
corpus  = manager.load_sentence_corpus()
lexical = manager.load_lexical_corpus()
train   = manager.load_train_split()
val     = manager.load_val_split()
test    = manager.load_test_split()

print(f"  Master Sentence Corpus : {len(corpus):,} multilingual concepts")
print(f"  Master Lexical Corpus  : {len(lexical):,} dictionary entries")
print(f"  Train Split            : {len(train):,} concepts (80%)")
print(f"  Val   Split            : {len(val):,} concepts (10%)")
print(f"  Test  Split            : {len(test):,} concepts (10%)")

print("\nRunning 0% leakage audit...")
checker = DataLeakageChecker(manager)
checker.verify_all()

print("\n" + "=" * 60)
print("✅  0% DATA LEAKAGE CONFIRMED")
print("✅  MASTER CORPUS LOADED")
print("✅  SETUP COMPLETE!")
print("=" * 60)
print("\n📋 Open any research notebook:")
print("  → notebooks/05_instruction_generation.ipynb")
print("  → notebooks/07_train_aya.ipynb")
print("  → notebooks/08_train_llama.ipynb")
print("  → notebooks/09_translation_evaluation.ipynb")